# VAE-GMM training on Colab: main model V1 (L = 64, β = 1e-4, critic on, 300 epochs)

Reproduction of Park et al., IEEE Access 14 (2026). The model code is the project's tested package (`whistle_code.zip`);
this notebook only sets up the session and runs it.

**Before the first session** (on the laptop, from the project root): `python colab/make_colab_bundle.py` writes
`colab/dist/whistle_code.zip`. Upload it to `whistle_vaegmm/code/` in Google Drive, and the dataset to
`whistle_vaegmm/data/preprocessed/`: `dataset_info.json` plus `encounter_*/images.npy` and `images.csv` of every encounter
(the Drive folder is set in cell 1; cells 3 and 5 check these files).

**Every session:** run cells 1–6. **First session only:** cells 7–8 (checks + 3-min timing benchmark). **Train:** cell 9.
After a disconnect, rerun cells 1–6 and then cell 9; training continues from the last finished epoch (checkpoints are on Drive).

Runtime: *Runtime → Change runtime type → A100 GPU*. Colab Pro has no background execution: keep this tab open.

In [ ]:
# ===== 1. Settings (the only cell to edit) =====
DRIVE_PROJECT   = "/content/drive/MyDrive/whistle_vaegmm"   # the folder uploaded to Google Drive
RUN_ID          = "V1_L64_b1e-4_critic"                     # fallback run: "V1_L64_b1e-4_critic_B896" with BATCH = 896
BATCH           = "auto"   # "auto" = 5376 on an 80 GB A100, 2688 on a 40 GB A100, else 896 (paper); or an integer
MICRO_BATCH     = 0        # 0 = whole batch at once; set e.g. half the batch printed by cell 2 if cell 8 runs out of GPU memory
TF32            = False    # True = faster tensor-core mode (small precision deviation, recorded in the run)
CUDNN_BENCHMARK = True     # cuDNN autotuning: same maths, faster kernels
CU_PER_HOUR     = 11.8     # approximate A100 compute units per hour (check your Colab plan page)

PAPER_BATCH, PAPER_LR_VAE, PAPER_LR_CRITIC = 896, 1e-3, 2e-4   # paper, App. A

In [ ]:
# ===== 2. GPU, RAM, Python; batch size and learning rates =====
import sys, subprocess, psutil, torch
assert sys.version_info >= (3, 11), f"needs Python >= 3.11, this runtime has {sys.version}"
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 GPU"
props = torch.cuda.get_device_properties(0)
gpu_gib = props.total_memory / 2**30
print(f"GPU {props.name} {gpu_gib:.1f} GiB | RAM {psutil.virtual_memory().total / 2**30:.1f} GiB | "
      f"Python {sys.version.split()[0]} | torch {torch.__version__}")
if BATCH == "auto":
    BATCH_SIZE = 5376 if gpu_gib >= 79 else 2688 if gpu_gib >= 39 else PAPER_BATCH
else:
    BATCH_SIZE = int(BATCH)
LR_VAE = PAPER_LR_VAE * BATCH_SIZE / PAPER_BATCH          # linear scaling rule (no deviation at 896)
LR_CRITIC = PAPER_LR_CRITIC * BATCH_SIZE / PAPER_BATCH
print(f"batch {BATCH_SIZE} | lr VAE {LR_VAE:.6g} | lr critic {LR_CRITIC:.6g}")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# ===== 3. Google Drive =====
import os
from google.colab import drive
drive.mount("/content/drive")
for rel in ("code/whistle_code.zip", "data/preprocessed/dataset_info.json"):
    assert os.path.exists(f"{DRIVE_PROJECT}/{rel}"), f"missing in Drive: {DRIVE_PROJECT}/{rel}"
print("Drive OK:", sorted(os.listdir(DRIVE_PROJECT)))

In [ ]:
# ===== 4. Code: unpack a fresh copy every session =====
import importlib.util, json, shutil, zipfile
CODE = "/content/whistle_code"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_PROJECT}/code/whistle_code.zip") as z:
    z.extractall("/content")
if importlib.util.find_spec("soundfile") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
if f"{CODE}/src" not in sys.path:
    sys.path.insert(0, f"{CODE}/src")
version = json.load(open(f"{CODE}/CODE_VERSION.json"))
print({k: v for k, v in version.items() if k != "files_sha256"})

In [ ]:
# ===== 5. Data: copy from Drive to the fast local disk (files already copied are skipped) =====
import glob, time
SRC, DST = f"{DRIVE_PROJECT}/data/preprocessed", "/content/preprocessed"
needed = ["dataset_info.json"] + [f"{os.path.basename(d)}/{f}" for d in sorted(glob.glob(f"{SRC}/encounter_*"))
                                  for f in ("images.npy", "images.csv")]
t0 = time.time()
for rel in needed:
    s, d = f"{SRC}/{rel}", f"{DST}/{rel}"
    if not os.path.exists(s):
        raise FileNotFoundError(f"missing in Drive: data/preprocessed/{rel}")
    if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
        continue
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copyfile(s, d)
    print(f"copied {rel} ({os.path.getsize(d) / 1e9:.2f} GB)")
from whistlepipe.vaegmm.data import WhistleImages
images = WhistleImages(DST)                    # checks every array against dataset_info.json
print(f"{len(images):,} images from {len(images.enc_dirs)} encounters ready ({time.time() - t0:.0f} s)")

In [ ]:
# ===== 6. Run configuration (paper values are locked inside configs/vaegmm.toml) =====
from whistlepipe.vaegmm.config import load_vaegmm_config
from whistlepipe.vaegmm.train import run_spec
steps_per_epoch = -(-len(images) // BATCH_SIZE)
overrides = [
    f'paths.dataset_dir="{DST}"',
    f'paths.runs_dir="{DRIVE_PROJECT}/runs"',
    f"deviations.batch_size={BATCH_SIZE}",
    f"deviations.lr_vae={LR_VAE!r}",
    f"deviations.lr_critic={LR_CRITIC!r}",
    'train.device="cuda"',
    "train.prefetch=2",
    f"train.log_every={max(1, steps_per_epoch // 4)}",
    f"train.micro_batch={MICRO_BATCH}",
    f"train.cudnn_benchmark={str(CUDNN_BENCHMARK).lower()}",
    f"train.tf32={str(TF32).lower()}",
]
cfg = load_vaegmm_config(f"{CODE}/configs/vaegmm.toml", overrides)
spec = run_spec(cfg, RUN_ID)                   # defaults = paper main model: L 64, beta 1e-4, critic, 300 epochs
print(json.dumps(spec, indent=1))
print("deviations:", cfg.to_dict()["deviations"], "| steps per epoch:", steps_per_epoch)

In [ ]:
# ===== 7. First session: data integrity (G1) and GPU numerics (G2), ~2 min =====
from whistlepipe.vaegmm.gates import gate_data, gate_numerics
g1, g2 = gate_data(cfg), gate_numerics(cfg)
assert g1["passed"] and g2["passed"], f"check failed, see {DRIVE_PROJECT}/runs/_gates"

In [ ]:
# ===== 8. First session: timing benchmark (~3 min, ~0.6 compute units; its model is discarded) =====
from whistlepipe.detector.ridgenet import set_deterministic
from whistlepipe.vaegmm.data import Prefetcher, epoch_plan
from whistlepipe.vaegmm.train import Trainer, apply_runtime_flags
set_deterministic()
apply_runtime_flags(cfg, spec)
tr = Trainer({**spec, "run_id": "benchmark"}, images, torch.device("cuda"), MICRO_BATCH)
vb, cb = epoch_plan(len(images), BATCH_SIZE, spec["n_critic"], spec["seed"], 1)
WARM, MEAS = 3, 12
pf = Prefetcher(images, vb[:WARM + MEAS], cb[:WARM + MEAS], depth=2)
g = tr.epoch_generator(1)
torch.cuda.reset_peak_memory_stats()
try:
    for i, (cbs, xb) in enumerate(pf):
        if i == WARM:
            torch.cuda.synchronize()
            t0, w0 = time.perf_counter(), pf.wait_s
        tr.step(cbs, xb, g)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(f"out of GPU memory at batch {BATCH_SIZE}: set MICRO_BATCH = {BATCH_SIZE // 2} in cell 1, rerun cells 1, 2, 6 and 8")
finally:
    pf.close()
s_it = elapsed / MEAS
hours = s_it * steps_per_epoch * spec["epochs"] / 3600
peak = torch.cuda.max_memory_allocated() / 2**30
print(f"{s_it:.2f} s/iteration | {s_it * steps_per_epoch / 60:.1f} min/epoch | {hours:.1f} h for {spec['epochs']} epochs "
      f"| ~{hours * CU_PER_HOUR:.0f} compute units")
print(f"peak GPU memory {peak:.1f} of {gpu_gib:.1f} GiB | data-loader wait {100 * (pf.wait_s - w0) / elapsed:.1f} % of the time")
if peak > 0.93 * gpu_gib:
    print("WARNING: close to the memory limit; consider MICRO_BATCH = BATCH_SIZE // 2")
del tr
torch.cuda.empty_cache()

### Cell 9: training

* Prints 4 log lines per epoch and one epoch summary. A non-finite loss stops the run with an error.
* Checkpoints: `runs/<RUN_ID>/checkpoints/last.pt` (every epoch), `epoch_025.pt` … (every 25), `final.pt` (end).
* **Resume:** rerun cells 1–6, then cell 9. If Colab gives a different A100 size in a later session, set `BATCH`
  in cell 1 to the batch size of the first session (the run refuses to continue with a different configuration).
* **Fallback if training is unstable** (error, or blank samples after ~20 epochs, see cell 10): set
  `RUN_ID = "V1_L64_b1e-4_critic_B896"`, `BATCH = 896` (paper's exact batch and learning rates) and start again from cell 1.

In [ ]:
# ===== 9. Train (resumable) =====
from whistlepipe.vaegmm.train import train_vae
last = f"{DRIVE_PROJECT}/runs/{RUN_ID}/checkpoints/last.pt"
if os.path.exists(last):
    print("resuming after epoch", torch.load(last, map_location="cpu", weights_only=False)["epoch"])
out = train_vae(cfg, spec, resume=True)
print("finished:", out)

In [ ]:
# ===== 10. Progress: loss curves, time left, latest sample grid (run when training is stopped or finished) =====
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
run = f"{DRIVE_PROJECT}/runs/{RUN_ID}"
recs = [json.loads(l) for l in open(f"{run}/train_log.jsonl")]
ep = pd.DataFrame([r for r in recs if r.get("epoch_summary")])
cols = [c for c in ("rec", "kl", "adv", "loss_d", "d_real", "d_fake", "gp") if c in ep]
ep.plot(x="epoch", y=cols, subplots=True, layout=(-1, 2), figsize=(12, 2 * ((len(cols) + 1) // 2)))
plt.tight_layout()
plt.show()
done, per = int(ep.epoch.max()), float(ep.epoch_s.tail(5).mean())
left_h = (spec["epochs"] - done) * per / 3600
print(f"epoch {done}/{spec['epochs']} | {per / 60:.1f} min/epoch | ~{left_h:.1f} h left (~{left_h * CU_PER_HOUR:.0f} compute units)")
pngs = sorted(glob.glob(f"{run}/samples_epoch_*.png"))
if pngs:
    display(Image(pngs[-1]))

In [ ]:
# ===== 11. After training: check the run, encode the latents (~1 min) =====
from whistlepipe.vaegmm.evaluate import encode
info = json.load(open(f"{DRIVE_PROJECT}/runs/{RUN_ID}/train_info.json"))
print(info["status"], "| final checkpoint SHA-256:", info.get("final_checkpoint_sha256"))
print(encode(cfg, RUN_ID))                     # latents_mu.npy (180,154 x 64): the GMM input

In [ ]:
# ===== 12. OPTIONAL: GMM fit here (~2 h, ~25 compute units). Default plan: fit it on the laptop (0 units). =====
RUN_GMM_HERE = False
if RUN_GMM_HERE:
    from whistlepipe.vaegmm.evaluate import fit_gmm
    print(fit_gmm(cfg, RUN_ID, 10000))

### After training

Download `whistle_vaegmm/runs/V1_L64_b1e-4_critic/` from Drive into the project's `models/vaegmm/` folder on the laptop,
then run the GMM fit and evaluation there (0 compute units):

```
python whistle.py fit-gmm --experiment V1 --K 10000
python whistle.py vae-eval recon --experiment V1
```